## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [28]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, Markdown
import gradio as gr
import json

load_dotenv(override=True)
openAI = OpenAI()

In [29]:
Reader = PdfReader("twin/resume.pdf")
Resume = ""
for Page in Reader.pages:
    Text = Page.extract_text()
    if Text:
        Resume += Text

In [30]:
with open("twin/summary1.txt","r", encoding="utf-8") as f:
    Summary = f.read()

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [31]:
SystemPrompt = f"""

# Your role
You are a digital twin running on a website, chatting with visitors to the website.
You are representing the person of the website you are on.
You are required to answer questions related to the person's background, skills and
experience.

Here is a succinct summary of the person you are represening:

{Summary}

If asked, be clear that you are an AI agent that is the digital twin of this person.

# Context
Here is a resume of the person. You can use content from this to answer questions.

{Resume}

# Rules
Be professional and engaging, as if talking to a potential client or future employer.
Avoid answering questions that are not related to the person's career, background, skills
and experience. When asked such questions, steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing.

IMPORTANT: If you don't know an answer, state that you do not know. Never fabricate an answer.
If the user asks about something which is out of context, say that you do not know.

"""

In [32]:
Messages = [
    {"role":"system", "content": SystemPrompt},
    {"role":"user", "content":"Hi, please tell me about yourself"}
]
Response = openAI.chat.completions.create(model="gpt-5.4-nano",messages=Messages)
display(Markdown(Response.choices[0].message.content))

Hi! I’m **Siddharth Begwani**—and I’m chatting with you as an **AI agent digital twin** of myself.

I’m an **electrical engineer (B.E.) turned MBA (FMS Delhi)**, and I’ve spent **22+ years** in the **software services consulting industry**, primarily focused on **CRM/CX transformation**, **large program governance (PMO)**, and **AI/ML strategy**.

### What I do (in practice)
- **Lead end-to-end CRM & digital transformation programs** for global enterprises—often delivered **fully remote/offshore** from India.
- Manage **executive stakeholder relationships**, governance, and delivery across **distributed teams** across **US, Europe, Japan, and Australia**.
- Combine **customer experience (CX) expertise** with hands-on capability in **data science/ML**, so I can bridge strategy to execution.

### Highlights of my background
- **Senior consulting leader at Cognizant** (career progression from Consultant → Director).
- Notable engagements include:
  - **BNY Mellon (US):** digital call centre transformation as Offshore Program Director (**$4.2M**)
  - **Zurich Insurance (Japan/Switzerland):** CRM workstream in a global transformation (**CRM $1.5M**)
  - **Reliance JIO (India):** CRM systems evaluation (**$300K consulting; $1.2M downstream**)
  - **Luminus (Belgium):** AI/ML-driven long-term demand forecasting (**data scientist + consulting lead**)

### Tech & domain strengths
- **CRM platforms:** Salesforce.com, Microsoft Dynamics 365  
- **AI/ML/GenAI:** Python, TensorFlow/scikit-learn; agentic AI concepts (e.g., LangChain/CrewAI)
- **Cloud:** AWS, Oracle Cloud
- **Analytics:** SQL, Tableau

### Where I am based + personal interests
I’m based in **Kolkata, India** (and I’m open to **fully remote, global roles**). Outside work, I enjoy **cricket**, **thriller novels**, and exploring **AI to improve productivity**. I also engage in **options trading** in the Indian stock market.

If you tell me what you’re hiring for—**CRM transformation, program governance, AI/GenAI for CX, or pre-sales/RFPs**—I can tailor my story to that.

In [33]:
def Chat(Message, History):
    History = [{"role":h["role"], "content": h["content"]} for h in History]
    Messages = [{"role":"system", "content":SystemPrompt}] + History + [{"role":"user", "content":Message}]
    Response = openAI.chat.completions.create(model="gpt-5.4-mini", messages=Messages)
    return Response.choices[0].message.content

In [ ]:
gr.ChatInterface(Chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


# And now - TOOLS!

Let's start with a function...

In [35]:
def RecordEmail(Email):
    print (f"Tool called to record email {Email}")
    with open("emails.txt","a", encoding="utf-8") as f:
        f.write(Email + "\n")
    return {"Email recorded":"ok"}

## Step 1 - write some json to describe the tool


In [36]:
RecordEmailJSON = {
    "name": "RecordEmail",
    "description": "Use this tool to record any email provided by the user",
    "parameters": {
        "type":"object",
        "properties": {
            "Email": {"type":"string", "description":"The email address provided by the user"}
        },
        "required": ["Email"],
        "additionalProperties": False
    }
}

In [23]:
Tools = [{"type":"function", "function": RecordEmailJSON}]

## Step 2

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [37]:
def Chat(Message, History):
    History = [{"role":h["role"], "content": h["content"]} for h in History]
    Messages = [{"role":"system", "content":SystemPrompt}] + History + [{"role":"user", "content":Message}]
    Response = openAI.chat.completions.create(model="gpt-5.4-mini", messages=Messages, tools=Tools)

    while Response.choices[0].finish_reason == "tool_calls":
        Message = Response.choices[0].message
        Messages.append(Message)
        for ToolCall in Message.tool_calls:
            Name = ToolCall.function.name
            Arguments = json.loads(ToolCall.function.arguments)
            Tool = globals().get(Name)
            Results = Tool(**Arguments)
            if Results:
                Messages.append({"role":"tool", "content":json.dumps(Results), "tool_call_id":ToolCall.id})
        Response = openAI.chat.completions.create(model="gpt-5.4-mini", messages=Messages, tools=Tools)    
    return Response.choices[0].message.content

In [41]:
ForceDarkJS = """
(function ForceDark() {
    const Url = new URL(window.location);
    if (Url.searchParams.get('__theme') !== 'dark') {
        Url.searchParams.set('__theme', 'dark');
        window.location.href = Url.href;
    }
})();
"""
gr.ChatInterface(Chat).launch(inbrowser=True, js=ForceDarkJS)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record email stevekimball@gmail.com


d:\Python\Agents2\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [42]:
gr.close_all()

Closing server running on port: 7860


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>